# 9.3 Inference in Bayesian networks: practice notebook

Get answers out of a Bayesian network. Sum exactly with enumeration and variable elimination, see why exact inference can be hard, and estimate with rejection sampling, likelihood weighting and Gibbs sampling.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Enumeration for P(Burglary | j, m)

In [ ]:
# Inference by enumeration on the burglary network (AIMA figure 14.9): P(Burglary | j, m).
parents = {"B": [], "E": [], "A": ["B", "E"], "J": ["A"], "M": ["A"]}
table = {"B": {(): 0.001}, "E": {(): 0.002},
         "A": {(True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001},
         "J": {(True,): 0.90, (False,): 0.05}, "M": {(True,): 0.70, (False,): 0.01}}
order = ["B", "E", "A", "J", "M"]

def prob(var, value, world):
    p = table[var][tuple(world[q] for q in parents[var])]
    return p if value else 1 - p

def enum_all(i, world):
    if i == len(order):
        return 1.0
    var = order[i]
    if var in world:                                   # evidence or query: one value
        return prob(var, world[var], world) * enum_all(i + 1, world)
    total = 0                                          # hidden: sum over both values
    for v in [True, False]:
        world[var] = v
        total = total + prob(var, v, world) * enum_all(i + 1, world)
    del world[var]
    return total

yes = enum_all(0, {"B": True, "J": True, "M": True})
no = enum_all(0, {"B": False, "J": True, "M": True})
print("unnormalised:", round(yes, 7), round(no, 7))
print("P(Burglary | j, m) =", round(yes / (yes + no), 3))

### Ten likelihood weighted samples

In [ ]:
# Likelihood weighting on the sprinkler network: estimate P(Rain | Sprinkler = true, WetGrass = true).
import random
rng = random.Random(1)
parents = {"C": [], "S": ["C"], "R": ["C"], "W": ["S", "R"]}
table = {"C": {(): 0.5}, "S": {(True,): 0.1, (False,): 0.5}, "R": {(True,): 0.8, (False,): 0.2},
         "W": {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}}
evidence = {"S": True, "W": True}

rain_w, total_w = 0, 0
for n in range(10):
    world, weight = {}, 1.0
    for var in ["C", "S", "R", "W"]:
        p = table[var][tuple(world[q] for q in parents[var])]
        if var in evidence:                    # fix the evidence and multiply in its likelihood
            world[var] = evidence[var]
            weight = weight * (p if evidence[var] else 1 - p)
        else:                                  # sample the others from their tables
            world[var] = rng.random() < p
    total_w = total_w + weight
    if world["R"]:
        rain_w = rain_w + weight
    print("sample", n + 1, "C =", world["C"], "R =", world["R"], "weight =", round(weight, 3))
print("estimate:", round(rain_w / total_w, 3), " exact: 0.3204")

## Your turn

### Task 1: Enumeration on the burglary network

Write `ask(evidence)` that returns P(B = True | evidence) by enumeration. evidence maps some of "J", "M" to True or False.

Hint: Write a recursive helper over ORDER. Fixed variables multiply in one entry, hidden ones add over True and False.

In [ ]:
PARENTS = {"B": [], "E": [], "A": ["B", "E"], "J": ["A"], "M": ["A"]}
TABLE = {"B": {(): 0.001}, "E": {(): 0.002},
         "A": {(True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001},
         "J": {(True,): 0.90, (False,): 0.05}, "M": {(True,): 0.70, (False,): 0.01}}
ORDER = ["B", "E", "A", "J", "M"]

def ask(evidence):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("round(ask({'J': True, 'M': True}), 3)", 0.284), ("round(ask({'J': True}), 4)", 0.0163), ("round(ask({'M': True}), 4)", 0.0561), ('round(ask({}), 4)', 0.001)])

### Task 2: Rejection sampling

Write `rejection(n, seed)` that draws n prior samples of the sprinkler network with `rng = random.Random(seed)`, keeps those with Sprinkler = True, and returns the fraction of kept samples with Rain = True.

Hint: Sample each variable in ORDER with rng.random() < P(true | parents). Count only samples with S True.

In [ ]:
import random
PARENTS = {"C": [], "S": ["C"], "R": ["C"], "W": ["S", "R"]}
TABLE = {"C": {(): 0.5}, "S": {(True,): 0.1, (False,): 0.5}, "R": {(True,): 0.8, (False,): 0.2},
         "W": {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}}
ORDER = ["C", "S", "R", "W"]

def rejection(n, seed):
    rng = random.Random(seed)
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('abs(rejection(20000, 1) - 0.3) < 0.03', True), ('abs(rejection(20000, 7) - 0.3) < 0.03', True)])

### Task 3: Likelihood weighting

Write `lw(n, seed)` that estimates P(Rain = True | Sprinkler = True, WetGrass = True) with n likelihood weighted samples.

Hint: For evidence variables, set the value and multiply the weight by its probability. Add weights, not counts.

In [ ]:
import random
PARENTS = {"C": [], "S": ["C"], "R": ["C"], "W": ["S", "R"]}
TABLE = {"C": {(): 0.5}, "S": {(True,): 0.1, (False,): 0.5}, "R": {(True,): 0.8, (False,): 0.2},
         "W": {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}}
ORDER = ["C", "S", "R", "W"]

def lw(n, seed):
    rng = random.Random(seed)
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('abs(lw(20000, 2) - 0.3204) < 0.03', True), ('abs(lw(20000, 5) - 0.3204) < 0.03', True)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: Enumeration on the burglary network

Mary alone raises P(b) more than John alone (0.056 against 0.016), because Mary has fewer false alarms (0.01 against 0.05).

In [ ]:
PARENTS = {"B": [], "E": [], "A": ["B", "E"], "J": ["A"], "M": ["A"]}
TABLE = {"B": {(): 0.001}, "E": {(): 0.002},
         "A": {(True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001},
         "J": {(True,): 0.90, (False,): 0.05}, "M": {(True,): 0.70, (False,): 0.01}}
ORDER = ["B", "E", "A", "J", "M"]

def prob(var, value, world):
    p = TABLE[var][tuple(world[q] for q in PARENTS[var])]
    if value:
        return p
    return 1 - p

def enum_all(i, world):
    if i == len(ORDER):
        return 1.0
    var = ORDER[i]
    if var in world:
        return prob(var, world[var], world) * enum_all(i + 1, world)
    total = 0
    for v in [True, False]:
        world[var] = v
        total = total + prob(var, v, world) * enum_all(i + 1, world)
    del world[var]
    return total

def ask(evidence):
    yes = dict(evidence)
    yes["B"] = True
    no = dict(evidence)
    no["B"] = False
    a = enum_all(0, yes)
    b = enum_all(0, no)
    return a / (a + b)

In [ ]:
check([("round(ask({'J': True, 'M': True}), 3)", 0.284), ("round(ask({'J': True}), 4)", 0.0163), ("round(ask({'M': True}), 4)", 0.0561), ('round(ask({}), 4)', 0.001)])

### Solution to task 2: Rejection sampling

The exact answer is 0.3, and about 70 percent of the samples are thrown away.

In [ ]:
import random
PARENTS = {"C": [], "S": ["C"], "R": ["C"], "W": ["S", "R"]}
TABLE = {"C": {(): 0.5}, "S": {(True,): 0.1, (False,): 0.5}, "R": {(True,): 0.8, (False,): 0.2},
         "W": {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}}
ORDER = ["C", "S", "R", "W"]

def rejection(n, seed):
    rng = random.Random(seed)
    kept = 0
    rain = 0
    for i in range(n):
        world = {}
        for var in ORDER:
            p = TABLE[var][tuple(world[q] for q in PARENTS[var])]
            world[var] = rng.random() < p
        if world["S"]:
            kept = kept + 1
            if world["R"]:
                rain = rain + 1
    return rain / kept

In [ ]:
check([('abs(rejection(20000, 1) - 0.3) < 0.03', True), ('abs(rejection(20000, 7) - 0.3) < 0.03', True)])

### Solution to task 3: Likelihood weighting

The exact answer 0.3204 comes from enumeration.

In [ ]:
import random
PARENTS = {"C": [], "S": ["C"], "R": ["C"], "W": ["S", "R"]}
TABLE = {"C": {(): 0.5}, "S": {(True,): 0.1, (False,): 0.5}, "R": {(True,): 0.8, (False,): 0.2},
         "W": {(True, True): 0.99, (True, False): 0.90, (False, True): 0.90, (False, False): 0.0}}
ORDER = ["C", "S", "R", "W"]

def lw(n, seed):
    rng = random.Random(seed)
    evidence = {"S": True, "W": True}
    rain = 0
    total = 0
    for i in range(n):
        world = {}
        weight = 1.0
        for var in ORDER:
            p = TABLE[var][tuple(world[q] for q in PARENTS[var])]
            if var in evidence:
                world[var] = evidence[var]
                if evidence[var]:
                    weight = weight * p
                else:
                    weight = weight * (1 - p)
            else:
                world[var] = rng.random() < p
        total = total + weight
        if world["R"]:
            rain = rain + weight
    return rain / total

In [ ]:
check([('abs(lw(20000, 2) - 0.3204) < 0.03', True), ('abs(lw(20000, 5) - 0.3204) < 0.03', True)])